# 1) Import libraries

- Pandas (might need OpenPYXL): Used for reading excel sheets into python-friendly objects. Use for large data replaced by duckdb.
- Pyarrow: Used to stream the export data into parquet or csv incase data is too large for duckdb's default export settings or a hardware bottleneck.
- RE(aka regex): Used to extract file identifiers from the file names.
- OS: Used to remove corrupted or partially saved files from a failed exporting attempt.
- IO: Used to wrap the binary CSV data in a StringIO wrapper to create a psuedofile that is readable by Duckdb
- DuckDB (might need fsspec): Used to load the data into relational tables, run sql queries for aggregation and exporting, passing the arrow RecordBatchReader to pyarrows for exporting, and for getting file names from the local directory.
- Requests: Used to query the download of data by the download URL passed to it.

Additional dependecies:
- OpenPYXL: Allows Pandas to load excel files.
- fsspec: Allows DuckDB to load StringIO wrapped CSV data.

In [1]:
import pandas as pd, pyarrow.parquet as pq, pyarrow.csv as csv
import re, os, io, duckdb, requests

# 2) Set Script Parameters



- Predownload parameters: allows user to define where and how the predownloaded files are named. It also lets the user define the export directory for Predownloaded-Files processing.
- Query parameters: allows the user to define the excel sheet where the Month_number, Year, CSV_URL columns are stored for the datasets. User can also define the export directory for Queried-Files processing.
- kept_columns: allows the user to prompt which columns they wish to keep in their output.
- spread_colums: allows the user to prompt which columns are spread out vertically to aggregate over and compress the data.
- Export parameters: allows the user to define the naming structure and the format of the exported files.
- Dim ParametersL allows the user to define the location of manually created dim excel file, to define the dim info to extract from the processing data, and where to export it to.

In [2]:
# Pick the years to include in the script
Years = range(2023,2026,1)

# Enter the parameters used in loading and storing files from the predownloaded process
predownloaded_directory = "monthly_datasets\\"
predownloaded_prefix = "hourly_transportation_"
predownloaded_suffix = ".csv" 
predownload_export = "export\\predownload\\"

# Enter the parameters used in loading and storing filed from the web-retireval process
query_list_file = "hourly_transportation_tracker.xlsx"
query_export = "export\\query\\"

# Enter the column names you wish to keep in your export, and which columns to aggregate accross
kept_columns = "transition_date, transition_hour, number_of_passage, number_of_passenger, product_kind, transaction_type_desc, town, line_name, station_poi_desc_cd"
spread_columns = "number_of_passage, number_of_passenger"

# Enter the parameters used in naming and storing the output files
export_prefix = "hourly_transportation_"
export_suffix = ".parquet"
export_format = "PARQUET"

# Enter the name of the database
db_name = "my_local_warehouse.duckdb"
date_col_name = "transition_date"

# Enter the parameters used in naming and storing the dim files
dims_address = "monthly_datasets\\DIMS.xlsx"  # for manually filled excel file
dims = {'line':   ["line", "line_name", "transport_type_id"],
        'road':   ["road_type", "transport_type_id"],
        'ticket': ["transfer_type", "transaction_type_desc"]}
dims_export = "export\\"

# Empty function to disable print
no_print = lambda *args, **kwargs: None


# 3) Define Helper Functions

### Duckdb relational table cleaner
Specified columns could have data entered over multiple entries instead of all together. This function merges spreadout entries to reduce the table's size.

In [171]:
# A function to remove extra columns and aggregate accross numeric columns which are spread out vertically
# ex:   "Vehicle" "Passangers"      "Vehicle" "Passangers"
#       Car         1           ->   Car        4
#       Car         3

def clean_dataset(dataset:duckdb.DuckDBPyRelation):
    # Fetch a list of columns to keep
    grouping_cols = [col_name.strip(", ") for col_name in kept_columns.split()]

    # Fetch a list of columns to aggregate accross by sum
    agging_dict = {col_name.strip(", "):"sum" for col_name in spread_columns.split()}
    
    # Ensure the grouping columns dont include the aggregation colums
    [grouping_cols.remove(col) for col in agging_dict.keys()]

    # Group by identity columns and compress the spread out information
    df_combined = duckdb.sql(f"""
        SELECT {", ".join(grouping_cols+[f"sum(TRY({col}::INTEGER)) as {col}" for col in spread_columns.split(", ")])}
        FROM dataset
        GROUP BY {", ".join(grouping_cols)}""")
    
    return df_combined


# test prompt
# file = duckdb.read_csv("monthly_datasets\\hourly_transportation_202410.csv")
# cleaning_test = clean_dataset(file)
# cleaning_test

### Duckdb table exporter
Primary method uses duckdb's sql COPY command.
Secondary method uses pyarrow's write_table functions.

In [172]:
# a function to handle exporting of the loaded data into a file
def save_to_file(dataset:duckdb.DuckDBPyRelation, export_address:str, kept_columns=kept_columns, print=print) -> bool:
    # ensure export address is a string
    if type(export_address) != str:
        print("export address must be a string!")
        raise TypeError
    
    try:
        # attempt export using duckdb
        print("exporting via duckdb...  ",end="")
        duckdb.sql(f"""COPY 
            (SELECT {kept_columns} from dataset) 
            TO "{export_address}"
            (FORMAT {export_format})""")
        
    except Exception as Error:
        print("Failed!")
        print("An error has occured in exporting using duckdb: ", Error)

        # remove partially saved/corrupted files
        if os.path.exists(export_address):
            os.remove(export_address)
            print("Cleaned up partial/corrupted file.")
        
        # attempt to export using pyarrows
        try:
            print("exporting via pyarrows...  ",end="")

            # 1. Fetch the query as an Arrow RecordBatchReader (streams data, doesn't load it all)
            reader = duckdb.sql(f"SELECT {kept_columns} FROM dataset").to_arrow_reader()

            # 2. Stream it straight into a Parquet or CSV file
            if export_format.lower() in ['parquet', 'pq']:
                pq.write_table(reader.read_all(), export_address)
            elif export_format.lower() in ['csv', 'txt']:
                csv.write_csv(reader.read_all(), export_address)
            else:
                raise ValueError(f"Unsupported file format: {export_format}")
            
        except Exception as Error:
            print("Failed!")
            print("An error has occured in exporting using pyarrows: ", Error)

            # remove partially saved/corrupted files
            if os.path.exists(export_address):
                os.remove(export_address)
                print("Cleaned up partial/corrupted file.")
                return False
            
    print("Done!")
    return True

### CSV Loader Function
This function takes either a local directory for a csv file or the binary data of a csv file wrapped in a StringIO wrapper. It loads the data of the csv file into a Duckdb relational table.

In [173]:
# a function to load csv data into a duckdb relational table
def load_from_csv(file_streamer:str|io.StringIO, file_name:str = "undefined") -> duckdb.DuckDBPyRelation:

    # load a file identifier to use in logging
    if type(file_streamer) == str:
        file_identifier = file_streamer.split("\\")[-1].split("/")[-1]
    elif type(file_streamer) == io.StringIO:
        try:
            str(file_name)
            file_identifier = file_name
        except Exception as e:
            raise e
    else:
        print("Invalid file streamer")
        raise TypeError
    
    ### extract file name and data
    try:
        print(file_identifier,"... ",sep="", end=" ")

        # load the csv data into a duckdb relation table
        file_db = duckdb.read_csv(file_streamer, filename=False)

    # terminate processing if file not found
    except IOError as error:
        print("Failed! File not found?")
        print(IOError)
        return False
    
    # terminate processing if unexpected error occurs
    except Exception as Error:
        print("Failed!")
        print("An unexpected error has occured in reading the file: ", Error)
        return False
    
    return file_db

### Export Files Finder Function
This function returns a pandas dataframe containing the exported files and the addresses they exist at.

In [174]:
# a function to collect the exported files and their addresses
def get_export_info() -> pd.DataFrame|bool:
    
    # Search for each valid year
    processed_file_directory_list = []
    for year in Years:
        # The naming format of exported files
        filename_pattern = f'{export_prefix}*{year}*{export_suffix}' 

        # Customize the directory based location based on user-defined export directories
        predownload_pattern = f'{predownload_export}{filename_pattern}'
        query_pattern = f'{query_export}{filename_pattern}'
        
        # Glob the files in each directory that fits the user-defined naming criteria
        processed_file_directory_list.extend([filename for filename, in duckdb.sql(f"""
            SELECT * FROM glob('{predownload_pattern}')
            UNION ALL
            SELECT * FROM glob('{query_pattern}')
            """).fetchall()])
        
    # return False if no files were found
    if not processed_file_directory_list:
        return False
    
    # Load the list of file addresses into a dataframe
    df = pd.DataFrame(processed_file_directory_list, columns=["File_address"])
    # Extract the file name
    df["File_name"] = df["File_address"].apply(lambda address: address.split("\\")[-1].split("/")[-1])
    # Extract the file directory
    df["File_directory"] = df.apply(lambda file: file["File_address"][0:-len(file["File_name"])], axis =1)

    # return a dataframe with 3 columns: File_name, predownload_export(contains address), query_export(contains address)
    return df.pivot(index="File_name", columns="File_directory", values="File_address")

# get_export_info()

### Date Table Compiler Function
A function to gather the recorded dates accross all data tables in the database and construct a dates table with columns for date, year, month, quarter, day, and weekday informations.

In [8]:
# a function to generate the date range from earliest to latest date or to scan all current dates directly.
# Using the date range, it builds a dates table in the database with extra information. (Year, Quarter, Month_Name, Month_No, Day, Weekday_Name, Weekday_No)
def date_loader(mode = "generator"):
    
    ### connect to the database
    my_db_con = duckdb.connect(db_name)

    ### data extraction from the database
    try:
        ## get table names to search through
        table_names = my_db_con.sql(f"SELECT * from (SELECT table_name from information_schema.tables WHERE table_name LIKE '{export_prefix}%')").fetchall()    
        date_from_all_query = " UNION ".join([f"SELECT DISTINCT {date_col_name} FROM {name}" for name, in table_names])

        ### fetch the usable dates based on mode
        match mode:
            ## scan the transition_date columns of all validly-named tables and compile all dates into a single column with no duplicates
            case "scanner":         
                # sql query string pattern for appending the transition_date columns. (tableA.transition_date Union tableB.transition_date Union ...)
                date_from_all_query = " UNION ".join([f"SELECT DISTINCT {date_col_name} FROM {name}" for name, in table_names])

                # sql query to return non-duplicate dates from across all validly-named tables
                table_dates = my_db_con.sql(f"SELECT DISTINCT {date_col_name}::DATE AS {date_col_name} from ({date_from_all_query}) ORDER BY {date_col_name};")                
                
            ## generate a date range base on the earliest and the latest dates found in the transition_date columns of all validly-named tables
            case "generator":
                # sql query string pattern for getting only the earliest and latest dates from the transition_date columns [tableA.transition_date.first,...] and [tableA.transition_date.last, ...] 
                first_date_from_all_query = " UNION ".join([f"SELECT {date_col_name} FROM {name} ORDER BY {date_col_name} ASC LIMIT 1" for name, in table_names])
                last_date_from_all_query = " UNION ".join([f"SELECT {date_col_name} FROM {name} ORDER BY {date_col_name} DESC LIMIT 1" for name, in table_names])

                # sql query to get the very first or last date from accross all transition_date columns of validly-named tables
                table_first_date = my_db_con.sql(f"SELECT {date_col_name}::DATE AS {date_col_name} from ({first_date_from_all_query}) ORDER BY {date_col_name} ASC LIMIT 1;")
                table_last_date = my_db_con.sql(f"SELECT {date_col_name}::DATE AS {date_col_name} from ({last_date_from_all_query}) ORDER BY {date_col_name} DESC LIMIT 1;")
                
                # sql query to generate a range of dates from earliest to lates date (inclusive)
                table_dates = my_db_con.sql(f"""SELECT CAST(range AS DATE) AS {date_col_name} 
                                            FROM range(
                                            (SELECT * FROM table_first_date LIMIT 1),
                                            (SELECT * FROM table_last_date LIMIT 1)+ INTERVAL 1 DAY,
                                            INTERVAL 1 DAY)""")
                
        ### extract additional date information from the date range
        table_date = my_db_con.sql(f"""
                                SELECT
                                    {date_col_name},
                                    CAST(date_part('year', {date_col_name}) AS INT) AS Year,
                                    'Q' || CAST(date_part('quarter', {date_col_name}) AS INT) AS Quarter,
                                    strftime({date_col_name}, '%B') AS Month_Name,
                                    CAST(date_part('month', {date_col_name}) AS INT) AS Month_No, 
                                    CAST(date_part('day', {date_col_name}) AS INT) AS Day,
                                    strftime({date_col_name}, '%A') AS Weekday_Name,
                                    CAST(date_part('isodow', {date_col_name}) AS INT) AS Weekday_No,
                                FROM table_dates""")
        ### write new 'dates' table to the database
        my_db_con.execute(f"CREATE OR REPLACE TABLE 'dates' AS SELECT * FROM table_date;")                     
    finally:    
        ### close the connection if finished or interupted
        my_db_con.close()
    return 

date_loader()

### Dim Extracting Function
This function extracts the dimensional data as defined by the user and stores it as an export for long term storage.

In [176]:
# a function to "pull" out the dim info from the given file
def pull_dims(file:duckdb.DuckDBPyRelation, file_id):
    
    # for each dimension
    for table, cols in dims.items():
        cols_joined = ", ".join(cols)
        
        # count the rows each unique set of data is found in the file
        dim_data = duckdb.sql(f"SELECT {cols_joined}, COUNT(*) AS count FROM file GROUP BY  {cols_joined}")

        # save it to a file without printing to the terminal
        dim_data_address = f'{dims_export}{file_id}{table}{export_suffix}'
        save_to_file(dim_data, dim_data_address, f"{cols_joined}, count", print=no_print)
    

### Dims Accountant Function
This function builds the dim_error table, which holds info on the presence of conflict based on the matching column's adjacent information. It also creates a dim_dim table which tables the strongest presence of the conflict.

In [177]:
# a function to produce the error dim table and a clean dim table
# error_vector represents how unclear the dim_info could be
def dims_accountant(db_conn:duckdb.DuckDBPyRelation, table, cols, cleaning_col, matching_col):
    cols_joined = ", ".join(cols)

    ### clean dirty string column
    temp = db_conn.sql(f""" SELECT {cols_joined}, sum(count) AS count 
                            FROM 
                                (SELECT 
                                    TRIM(REPLACE({cleaning_col}, '\"','')) AS {cols_joined}, SUM(count) AS count 
                                FROM 
                                    {table}_temp 
                                GROUP BY 
                                    {cols_joined})
                            GROUP BY 
                                {cols_joined}""")
    
    ### add error table
    db_conn.sql(f"""CREATE OR REPLACE TABLE '{table}_error' AS 
                        (SELECT 
                            {", ".join([f"t1.{col}" for col in cols])}, t1.count, t2.Error_Vector 
                        FROM 
                            (SELECT 
                                * 
                            FROM 
                                temp 
                            ) t1
                        LEFT JOIN 
                            (SELECT 
                                {matching_col}, count(*) AS Error_Vector 
                            FROM 
                                temp 
                            GROUP BY 
                                {matching_col}
                            ) t2
                        ON t1.{matching_col} = t2.{matching_col})
                        """)
    
    ### add highest occurences of each dim info into a clean table
    ranked_partition = db_conn.sql(f"SELECT {cols_joined}, count, DENSE_RANK() OVER (PARTITION BY {matching_col} ORDER BY count DESC) AS rank from {table}_error")
    db_conn.sql(f"CREATE OR REPLACE TABLE '{table}_dim' AS SELECT {cols_joined} FROM ranked_partition WHERE rank = 1;")


# 4) Load The Files and Store Them In Compressed Format

## Method 1: Use predownloaded files

### Defind the predownload function
It takes a local address, searches it for the csv data, processes it, and exports it.

In [178]:
# Function to fetch CSV files, using a download link, process it, and export it
def predownloaded_CSV(file_address):

    ### Ensure given address can only be a string
    if type(file_address)!= str:
        print("file_address must be a string")
        raise TypeError
    
    ### extract file name and data
    file_name = file_address.split("\\")[-1].split("/")[-1]  #alternate incase filename=True duckdb.sql("SELECT filename from file_db LIMIT 1").fetchall()[0][0]
    file_db = load_from_csv(file_streamer = file_address)
    if not file_db:
        print("No data Found!")
        return False
    
    ### Positive response initiates export name generation
    # get a name id mimic for the export file 
    match = re.findall(f"{predownloaded_prefix}(.*){predownloaded_suffix}", file_name)
    export_id = match[0] if match != [] else ""
    
    # generate the export address 
    export_address = f"{predownload_export}{export_prefix}{export_id}{export_suffix}"

    ### pull the dims
    try:
        print("Pulling Dims...  ",end="")
        pull_dims(file_db, export_id)
    except Exception as Error:
        print("Failed!")
        print("An error has occured in pulling the dims: ", Error)
        return False
    
    ### clean the duckdb table
    try:
        print("cleaning...  ",end="")
        cleaned_db = clean_dataset(file_db)
    except Exception as Error:
        print("Failed!")
        print("An error has occured in cleaning the table: ", Error)
        return False

    ### export the cleaned table
    return save_to_file(cleaned_db, export_address)


### Collecting valid files as defined by the user's parameters

In [179]:
# Fetch the list of files that were manually saved as {predownloaded_suffix} for the selected years
predownloaded_file_directory_list = []
for year in Years:
    filename_pattern = f'{predownloaded_directory}{predownloaded_prefix}*{year}*{predownloaded_suffix}' 
    predownloaded_file_directory_list.extend([filename for filename, in duckdb.sql(f"SELECT * FROM glob('{filename_pattern}')").fetchall()])

### Perform the processing of only the predownloaded files

In [185]:
def process_all_preloaded():
    if  not predownloaded_file_directory_list:
        print("There are no files to process")
    else:
        processing_status = [predownloaded_CSV(file_address) for file_address in predownloaded_file_directory_list[-5:]]
        print(processing_status, sep="\n")

process_all_preloaded()

hourly_transportation_202302.csv...  

Pulling Dims...  cleaning...  exporting via duckdb...  Done!
hourly_transportation_202305.csv...  Pulling Dims...  cleaning...  exporting via duckdb...  Done!
hourly_transportation_202306.csv...  Pulling Dims...  cleaning...  exporting via duckdb...  Done!
hourly_transportation_202307.csv...  Pulling Dims...  cleaning...  exporting via duckdb...  Done!
hourly_transportation_202308.csv...  Pulling Dims...  cleaning...  exporting via duckdb...  Done!
hourly_transportation_202309.csv...  Pulling Dims...  cleaning...  exporting via duckdb...  Done!
hourly_transportation_202310.csv...  Pulling Dims...  cleaning...  exporting via duckdb...  Done!
hourly_transportation_202311.csv...  Pulling Dims...  cleaning...  exporting via duckdb...  Done!
hourly_transportation_202312.csv...  Pulling Dims...  cleaning...  exporting via duckdb...  Done!
hourly_transportation_202401.csv...  Pulling Dims...  cleaning...  exporting via duckdb...  Done!
hourly_transportation_202402.csv...  Pulling Dims...  cle

## Method 2: Querying a request for the files

### Define the query function
It takes a row-based query - must contain Month_number, Year, and CSV_URL columns, sends a GET request it for the csv data, loads the data with a IO.StringIO wrapper, processes it, and exports it.

In [ ]:
# Function to fetch CSV files, using a download link, process it, and export it
def query_CSV(query):
    print(query.CSV_URL)
    ### query the source for the CSV File
    response = requests.get(query.CSV_URL, headers = {
    "Connection": "close"
})

    ### If response is negative, print error message
    if response.status_code != 200:
        print(f'{query.Month} {query.Year} returned a code of {response.status_code}')
        return False
    
    ### Positive response initiates file processing
    else:
        # generate naming id
        export_id = f'{query.Year}{query.Month_number:02d}'
        # generate the export address 
        export_address = f"{query_export}{export_prefix}{export_id}{export_suffix}"

        ### extract file data
        try:
            pseudofile = io.StringIO(response.text)
        except Exception as e:
            print("failed to stream the retrieved data into a table")
            print(e)
            return False
            
        ### load the csv data into a duckdb relation table 
        file_identifier = f"{export_prefix}{export_id}{predownloaded_suffix}"
        file_db = load_from_csv(file_streamer = pseudofile, file_name=file_identifier)
        if not file_db:
            print("No data found!")
            return False
        
         ### pull the dims
        try:
            print("Pulling Dims...  ",end="")
            pull_dims(file_db, export_id)
        except Exception as Error:
            print("Failed!")
            print("An error has occured in pulling the dims: ", Error)
            return False
        
        ### clean the duckdb table
        try:
            print("cleaning...  ",end="")
            cleaned_db = clean_dataset(file_db)
        except Exception as Error:
            print("Failed!")
            print("An error has occured in loading and cleaning the table: ", Error)
            return False
        
        # export the cleaned table
        return save_to_file(cleaned_db, export_address)


### Load row-based queries from the user-defined excel workbook

In [18]:
# Fetch the spreadsheet with resource info in it
query_sheet = pd.read_excel(query_list_file)
# Filter resource sheet by selected years
filtered_query_sheet = query_sheet[query_sheet.Year.isin(Years)]


### Perform the processing of only the available queries

In [19]:
def process_all_queries():
    if filtered_query_sheet.empty:
        print("There are no files to query")
    else:
        processing_status = filtered_query_sheet.apply(query_CSV, axis=1)
        print(processing_status, sep="\n")

# process_all_queries()

## Method 3: Priority-based processing
An order is determine whether queries to the server or using predownloaded files should be considered first. Also permits the user to either reprocess the files or skip over already processed output files.

In [ ]:
# Function to iterate through the tracker excel file and run hybrid processing based on user-definable priority.
# User can also define whether to reprocess files or rely on preprocessed files
def priority_based_looker(row, priority_sorted_columns:list[str] = ['File_Path', 'CSV_URL'], reprocess = False): 

    # iterate over columns from highest priority first
    for column in priority_sorted_columns:
        # take first source and load it
        if pd.notna(row[column]):
            match column:
                # perform predownloaded processing unless reprocessing is undesired
                case 'File_Path':
                    if reprocess or pd.isna(row['Predownload_Export_Path']):
                        return predownloaded_CSV(row[column])
                    return "Already exists in exports"
                
                # perform query processing unless reprocessing is undesired
                case 'CSV_URL':
                    if reprocess or pd.isna(row['Query_Export_Path']):
                        return query_CSV(row)
                    return "Already exists in exports"
                
    # if a supported source was not found
    return "A supported source was not found"


if filtered_query_sheet.empty:
    print("There are no files to query")
else:
    processing_status = filtered_query_sheet.apply(lambda row: priority_based_looker(row, reprocess=False), axis=1)
    print(processing_status, sep="\n")



    



https://data.ibb.gov.tr/dataset/a6855ce7-4092-40a5-82b5-34cf3c7e36e3/resource/afc1cec1-fd43-44a9-b5c0-977d3b1d60d7/download/hourly_transportation_202301.csv
hourly_transportation_202301.csv...  cleaning...  exporting via duckdb...  Done!
hourly_transportation_202302.csv...  cleaning...  exporting via duckdb...  Done!
https://data.ibb.gov.tr/dataset/a6855ce7-4092-40a5-82b5-34cf3c7e36e3/resource/eb32254e-3233-4ac9-ade0-cd056ccfa509/download/hourly_transportation_202303.csv
hourly_transportation_202303.csv...  cleaning...  exporting via duckdb...  Done!
hourly_transportation_202306.csv...  cleaning...  exporting via duckdb...  Done!
hourly_transportation_202307.csv...  cleaning...  exporting via duckdb...  Done!
hourly_transportation_202308.csv...  cleaning...  exporting via duckdb...  Done!
hourly_transportation_202309.csv...  cleaning...  exporting via duckdb...  Done!
hourly_transportation_202310.csv...  cleaning...  exporting via duckdb...  Done!
hourly_transportation_202311.csv...  c

# 5) Load The DIMS and Store Them In Compressed Format

Aggregate the dim files for each file processed, build the error and clean tables, and save them for long term storage.

In [189]:
# a function to build the dim tables accross all processed data and save it for long term storage
def dims_exporter():
    # establish a connection in memory
    temp_conn = duckdb.connect()

    # for each dim
    for table, cols in dims.items():
        # read all validly-named dim exports
        table_data_address = f"{dims_export}*{table}{export_suffix}"        
        temp_conn.sql(f"CREATE OR REPLACE TABLE {table}_temp AS SELECT * FROM read_{export_format}('{table_data_address}')")
        
        # build and save table_error and table_dim
        try:
            dims_accountant(temp_conn, table, cols, cols[0], cols[1])
            temp_conn.sql(f"COPY {table}_dim to '{dims_export}{table}_dim{export_suffix}' (FORMAT {export_format})")
            temp_conn.sql(f"COPY {table}_error to '{dims_export}{table}_error{export_suffix}' (FORMAT {export_format})")
        except Exception as error:
            print(error)

    # close the connection
    temp_conn.close()
    
dims_exporter()

# 6) Load files into SQL database

### Priority Based SQL Loader
The directories of the export files are passed in a priority order and they are loaded into the database by tables partitioned either by Year, Month, or None (all in one table)

In [10]:
# a function to load exported files by highest priority first, and partition into tables by "Year", "Month", or "None" (all in one table)
def priority_based_sql_loader(priority_sorted_columns:list[str] = [predownload_export, query_export], reload = True, partition = None):
    # partition by month
    def load_files_by_month(row):
        # get table name and file path from row data
        table_name = row.name.split(".")[0]
        table_data = row["File_path"]
        
        # if user wants to reload the "monthly" tables, initialize new empty tables that can replace them if they exist
        if reload:
            my_db_con.execute(f"CREATE OR REPLACE TABLE '{table_name}' AS SELECT * from read_parquet('{table_data}')")

        # without editing existing tables
        else:
            # find existing tables
            table_exists = my_db_con.sql(f"SELECT EXISTS (SELECT 1 from information_schema.tables WHERE table_name = '{table_name}')").fetchall()[0][0]
            # only add non-existing tables
            if not table_exists:
                my_db_con.execute(f"CREATE TABLE '{table_name}' AS SELECT * from read_parquet('{table_data}')")
        return
    
    # partition by year
    def load_files_by_year(row):
        # get table name and file path from row data
        table_name = re.findall(f"{export_prefix}{'(\\d{4})\\d{2}'}{export_suffix}",row.name)
        if table_name:
            table_name = table_name[0]
        else:
            return False
        table_data = row["File_path"]

        # insert scanned data into it's yearly table
        my_db_con.execute(f"INSERT INTO '{table_name}' SELECT * from read_parquet('{table_data}')")
        return
    
    # no partition
    def load_files(row):
        # get file path from row data
        table_data = row["File_path"]
        
        # insert scanned data into the "all" table
        my_db_con.execute(f"INSERT INTO '{export_prefix}all' SELECT * from read_parquet('{table_data}')")
        return

    ### get the export information dataframe: File_name, predownload_export(contains address), query_export(contains address)
    export_info = get_export_info()
    
    ### if export_info was found, begin processing
    if type(export_info) == pd.DataFrame:
        # create a new empty column for File_path and populate it's empty slots with highest priority files first
        export_info["File_path"] = pd.NA
        for column in priority_sorted_columns:
            export_info.fillna({"File_path":export_info[column]}, inplace=True)
        
        if not export_info.empty:
            # establish a connection to the database
            my_db_con = duckdb.connect(db_name)

            try:
                # load data without a partition
                if partition == None:
                    # if user wants to reload the "all" table, initialize a new empty table that can replace it if it exists
                    if reload:                        
                        my_db_con.execute(f"CREATE OR REPLACE TABLE '{export_prefix}all' AS SELECT * FROM read_parquet('{export_info["File_path"].iloc[0]}') WHERE 1=0")
                    export_info.apply(load_files, axis=1)

                # load data with a monthly partition
                elif partition == "Month":
                    export_info.apply(load_files_by_month, axis=1)

                # load data with a yearly partition
                elif partition == "Year":   
                    # if user wants to reload the "yearly" tables, initialize new empty tables that can replace them if they exist
                    if reload:
                        for year in Years:
                            my_db_con.execute(f"CREATE OR REPLACE TABLE '{export_prefix}{year}' AS SELECT * FROM read_parquet('{export_info["File_path"].iloc[0]}') WHERE 1=0")                     
                    export_info.apply(load_files_by_year, axis=1)

                # incase user enters an unsupported partition type
                else:
                    print("bad partition")

            # incase any errors arrise
            except Exception as e:
                raise e
            
            # close the db connection on completion or interruption
            finally:
                my_db_con.close()
        # on changes to the tables, rebuild the dates table
        date_loader()
    
priority_based_sql_loader()

### Dimension Tables Loader
Reload the given dims tables that are stored in the excel sheet at the dims address

In [ ]:
#   a function to load the dim tables from sheet names of an excel sheet
def dims_loader(sheet_names = []):
    # ensure type of parameter
    sheet_names = list(sheet_names)
    # establish connection to the database
    my_db_con = duckdb.connect(db_name)

    # load each sheet into the database
    for sheet_name in sheet_names:
        if type(sheet_name) == str:
            # get data from excel using pandas
            dim_table = pd.read_excel(dims_address, sheet_name=sheet_name)
            # write a new dim table from the pandas dataframe
            my_db_con.execute(f"CREATE OR REPLACE TABLE '{sheet_name}' AS SELECT * FROM dim_table")                     

    # close the connection
    my_db_con.close()
    return 


# dims_loader(sheet_names=["ticket_dim", "line_dim", "road_dim"])

### Extracted Dimensions Loader
This function loads the Dim tables from exported files based on the raw data that was processed for the database

In [190]:
# a function to load the dims tables from previously exported files
def extracted_dims_loader():
    # establish a connection to the databse
    my_db_con = duckdb.connect(db_name)
    
    # for each dim
    for dim in dims.keys():

        # the regex pattern for finding validly named files
        # "\(" can cause an error in regex
        if dims_export.endswith("\\"):
            pattern = f"{dims_export}" + r"\(" f"{dim}.*){export_suffix}"
        else:
            pattern = f"{dims_export}({dim}.*){export_suffix}"

        # query the dim tables to load
        dim_file_addresses = [file_name for file_name, in duckdb.sql(f"SELECT * FROM glob('{dims_export}{dim}*{export_suffix}')").fetchall()]

        # for each table file
        for file_address in dim_file_addresses:

            # get the table name embedded in the file address
            table_name = re.findall(pattern, file_address)
            if table_name:
                table_name = table_name[0]
                print("Loading: ",table_name)
            else:
                continue
            # add the file data to the table in the database
            my_db_con.execute(f"CREATE OR REPLACE TABLE '{table_name}' AS SELECT * FROM read_{export_format}('{dims_export}{table_name}{export_suffix}')")

    # close the connection
    my_db_con.close()

extracted_dims_loader()

Loading:  line_dim
Loading:  line_error
Loading:  road_dim
Loading:  road_error
Loading:  ticket_dim
Loading:  ticket_error


# 7) Run Local QUACK SQL Server

### Launch the QUACK Server
Using QUACK to run the database, we can query it using sql queries with DUCKDB attachments or Quack Connectors.

In [8]:
server = duckdb.connect(db_name)
server.sql("CALL quack_serve('quack:localhost', token = 'super_secret');")

┌─────────────────┬───────────────────────┬──────────────┐
│   listen_uri    │      listen_url       │  auth_token  │
│     varchar     │        varchar        │   varchar    │
├─────────────────┼───────────────────────┼──────────────┤
│ quack:localhost │ http://localhost:9494 │ super_secret │
└─────────────────┴───────────────────────┴──────────────┘

### Close the QUACK Server
Safely close the Quack server when it is no longer needed or to restart.

In [9]:
server.sql("CALL quack_stop('quack:localhost');")
server.close()

In [146]:
db = duckdb.connect(db_name)
# db.sql("SELECT * FROM line_error").show()
db.sql("SELECT * FROM read_parquet('export\\*09line.parquet')").show()
db.close()

┌────────────────────────────────────┬────────────┬───────────────────┬────────┐
│                line                │ line_name  │ transport_type_id │ count  │
│              varchar               │  varchar   │       int64       │ int64  │
├────────────────────────────────────┼────────────┼───────────────────┼────────┤
│ KABATAS-MAHMUTBEY                  │ M7         │                 2 │ 459226 │
│ KIRAZLI-BASAKSEHIR/METROKENT       │ M3         │                 2 │ 195113 │
│ KAYASEHIR-BAKIRKOY                 │ 79B        │                 1 │  33168 │
│ YENIBOSNA METRO-ARNAVUTKOY         │ 36AY       │                 1 │  20130 │
│ BASAKSEHIR 4 -1. ETAPLAR - TAKSIM  │ 89C        │                 1 │  31242 │
│ KEMERBURGAZ-GOKTURK-ARNAVUTKOY     │ 48KA       │                 1 │  16279 │
│ OMERLI KIPTAS KONUTLARI-BEYLIKDUZU │ 144A       │                 1 │  16986 │
│ ASIYAN-USKUDAR                     │ ASIYAN-USK │                 3 │  12122 │
│ ADALAR-KABATAS            